In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print(f"CUDA: {torch.version.cuda}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

NVIDIA A100-SXM4-80GB
CUDA: 12.8
VRAM: 85.1 GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Chewdoughsay/ViT-FP8-experiments.git
%cd ViT-FP8-experiments

Mounted at /content/drive
Cloning into 'ViT-FP8-experiments'...
remote: Enumerating objects: 338, done.
remote: Counting objects: 100% (338/338), done.
remote: Compressing objects: 100% (233/233), done.
remote: Total 338 (delta 153), reused 283 (delta 98), pack-reused 0 (from 0)
Receiving objects: 100% (338/338), 11.79 MiB | 17.50 MiB/s, done.
Resolving deltas: 100% (153/153), done.
/content/ViT-FP8-experiments


In [ ]:
!pip install -q timm datasets huggingface_hub

In [ ]:
import shutil, os

os.makedirs('data/imagenet-1k', exist_ok=True)
shutil.copytree(
    '/content/drive/MyDrive/imagenet-1k',  
    'data/imagenet-1k',
    dirs_exist_ok=True
)
print("Dataset copiat OK")
!ls data/imagenet-1k/

Dataset copiat OK
validation-00000-of-00014.parquet  validation-00007-of-00014.parquet
validation-00001-of-00014.parquet  validation-00008-of-00014.parquet
validation-00002-of-00014.parquet  validation-00009-of-00014.parquet
validation-00003-of-00014.parquet  validation-00010-of-00014.parquet
validation-00004-of-00014.parquet  validation-00011-of-00014.parquet
validation-00005-of-00014.parquet  validation-00012-of-00014.parquet
validation-00006-of-00014.parquet  validation-00013-of-00014.parquet


In [ ]:
import sys
sys.path.insert(0, '/content/ViT-FP8-experiments')

import timm, timm.data, torch
from src.models.quantized_linear import quantize_model_selective

model = timm.create_model('vit_tiny_patch16_224', pretrained=True).cuda().eval()
model, stats = quantize_model_selective(model, verbose=False)
x = torch.randn(4, 3, 224, 224).cuda()
with torch.no_grad():
    out = model(x)
print(f"Forward pass OK — {out.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/22.9M [00:00<?, ?B/s]

Forward pass OK — torch.Size([4, 1000])


In [ ]:
!python scripts/phase4/evaluate_model.py \
    --model vit_tiny_patch16_224 \
    --batch-size 128

Phase 4 — vit_tiny_patch16_224  (5.7M params)
PyTorch 2.10.0+cu128  |  device: cuda

Loading model...
  Data config: {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'crop_pct': 0.9, 'crop_mode': 'center'}

Loading ImageNet-1k validation...
  Loading from local parquet: data/imagenet-1k
Generating train split: 50000 examples [00:16, 3012.88 examples/s]
  50,000 images


STEP 1: FP32 baseline
  Memory: 21.81 MB
FP32: 100% 391/391 [01:41<00:00,  3.87it/s]
  Accuracy: 75.454%  |  Latency: 29.641 ms/batch

STEP 2: FP16 (model.half())
  Memory: 10.905 MB  (reduction: 2.00x)
FP16: 100% 391/391 [01:42<00:00,  3.82it/s]
  Accuracy: 75.458%  |  Latency: 8.956 ms/batch

STEP 3: INT8 per-tensor
  Memory: 6.623 MB  (reduction: 3.29x)
INT8-pt: 100% 391/391 [01:40<00:00,  3.89it/s]
  Accuracy: 75.128%  |  Avg MSE: 2.88e-06

STEP 4: INT8 per-channel
INT8-pc: 100% 391/391 [01:40<00:00,  3.88it/s]
  Accuracy: 75.418%  |  Memory: 6.702 MB

SUMMAR

In [ ]:
import shutil
shutil.copytree(
    '/content/ViT-FP8-experiments/results/Phase4',
    '/content/drive/MyDrive/vit_experiments/results/Phase4',
    dirs_exist_ok=True
)
print("Rezultate salvate pe Drive!")

Rezultate salvate pe Drive!


In [ ]:
!python scripts/phase4/evaluate_model.py \
    --model vit_small_patch16_224 \
    --batch-size 128

Phase 4 — vit_small_patch16_224  (22M params)
PyTorch 2.10.0+cu128  |  device: cuda

Loading model...
model.safetensors: 100% 88.2M/88.2M [00:02<00:00, 39.7MB/s]
  Data config: {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'crop_pct': 0.9, 'crop_mode': 'center'}

Loading ImageNet-1k validation...
  Loading from local parquet: data/imagenet-1k
  50,000 images


STEP 1: FP32 baseline
  Memory: 84.117 MB
FP32: 100% 391/391 [01:41<00:00,  3.87it/s]
  Accuracy: 81.382%  |  Latency: 73.09 ms/batch

STEP 2: FP16 (model.half())
  Memory: 42.058 MB  (reduction: 2.00x)
FP16: 100% 391/391 [01:41<00:00,  3.84it/s]
  Accuracy: 81.394%  |  Latency: 15.038 ms/batch

STEP 3: INT8 per-tensor
  Memory: 23.367 MB  (reduction: 3.60x)
INT8-pt: 100% 391/391 [01:41<00:00,  3.85it/s]
  Accuracy: 81.296%  |  Avg MSE: 3.08e-06

STEP 4: INT8 per-channel
INT8-pc: 100% 391/391 [01:41<00:00,  3.87it/s]
  Accuracy: 81.396%  |  Memory: 23.525 MB

SUMMARY
Me

In [ ]:
import shutil
shutil.copytree(
    '/content/ViT-FP8-experiments/results/Phase4',
    '/content/drive/MyDrive/vit_experiments/results/Phase4',
    dirs_exist_ok=True
)
print("Rezultate salvate pe Drive!")

Rezultate salvate pe Drive!


In [ ]:
!python scripts/phase4/evaluate_model.py \
    --model vit_base_patch16_224 \
    --batch-size 64

Phase 4 — vit_base_patch16_224  (86M params)
PyTorch 2.10.0+cu128  |  device: cuda

Loading model...
model.safetensors: 100% 346M/346M [00:03<00:00, 101MB/s]
  Data config: {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'crop_pct': 0.9, 'crop_mode': 'center'}

Loading ImageNet-1k validation...
  Loading from local parquet: data/imagenet-1k
  50,000 images


STEP 1: FP32 baseline
  Memory: 330.229 MB
FP32: 100% 782/782 [01:45<00:00,  7.41it/s]
  Accuracy: 85.102%  |  Latency: 128.772 ms/batch

STEP 2: FP16 (model.half())
  Memory: 165.115 MB  (reduction: 2.00x)
FP16: 100% 782/782 [01:43<00:00,  7.59it/s]
  Accuracy: 85.11%  |  Latency: 18.235 ms/batch

STEP 3: INT8 per-tensor
  Memory: 87.229 MB  (reduction: 3.79x)
INT8-pt: 100% 782/782 [01:45<00:00,  7.40it/s]
  Accuracy: 85.08%  |  Avg MSE: 1.34e-06

STEP 4: INT8 per-channel
INT8-pc: 100% 782/782 [01:45<00:00,  7.40it/s]
  Accuracy: 85.1%  |  Memory: 87.546 MB

SUMMARY
Method

In [ ]:
import shutil
shutil.copytree(
    '/content/ViT-FP8-experiments/results/Phase4',
    '/content/drive/MyDrive/vit_experiments/results/Phase4',
    dirs_exist_ok=True
)
print("Rezultate salvate pe Drive!")

Rezultate salvate pe Drive!
